In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

## Exercise 1: Sparse optical flow

In [6]:
cap = cv2.VideoCapture('Robots.mp4')
cv2.namedWindow('output_frame', cv2.WINDOW_NORMAL)

In [ ]:
ret, old_frame = cap.read()
b,g,r = cv2.split(old_frame) # Changing the order from bgr to rgb so that matplotlib can show it
old_frame = cv2.merge([r,g,b])
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_RGB2GRAY)
#plt.imshow(old_gray, cmap = 'gray')
old_feat = cv2.goodFeaturesToTrack(old_gray, maxCorners=300, qualityLevel=0.3, minDistance=7)
mask = np.zeros_like(old_frame)
while True:
    ret, new_frame = cap.read()
    if not ret:
        break
    b,g,r = cv2.split(new_frame) # Changing the order from bgr to rgb so that matplotlib can show it
    new_frame = cv2.merge([r,g,b])
    new_gray = cv2.cvtColor(new_frame, cv2.COLOR_RGB2GRAY)
    #plt.imshow(new_gray, cmap = 'gray')
    new_feat, status, error = cv2.calcOpticalFlowPyrLK(old_gray, new_gray, old_feat, None)
    if new_feat is not None:
        valid_new = []
        mask = (mask * 0.90).astype(np.uint8)
        for i in range(len(old_feat)):
            if status[i] == 1:
                f10 = int(old_feat[i][0][0])
                f11 = int(old_feat[i][0][1])
                f20 = int(new_feat[i][0][0])
                f21 = int(new_feat[i][0][1])
                mask = cv2.line(mask, (f10, f11), (f20, f21), (0, 255, 0), 2)
                new_frame = cv2.circle(new_frame, (f20, f21), 5, (255, 0, 0), -1)
                valid_new.append(new_feat[i])

        output_frame = cv2.add(new_frame, mask)
        old_frame=new_frame
        old_gray=new_gray
        old_feat=np.array(valid_new)
        cv2.imshow('output_frame', output_frame)
        cv2.waitKey(20)
    else:
        break

cv2.destroyAllWindows()
cv2.waitKey(1)


KeyboardInterrupt: 

: 

In the first exercise we use the functions learned in Exercise 4, but as we are working with a video, we have to compare not only two images but keep comparing two images until the video finishes. The way we analyse the images is no different from the previous exercise. In order to see where the robot has moved we create a mask, instead of drawing paths directly onto incoming frames because they would be deleted in each iteration. In order to have an idea of the movement we multiply the mask's pixel values by a fractional factor (0.90) at each iteration so the older points fade away. Apart from that we initialize an empty list called: valid_new = [], in wich we include detected points with a status value of 1, which are the "valid" ones.
In the video we can appreciate that some detected points get lost, but in general the result shows pretty good trajectories.

## Dense optical flow

### Helper functions for different representations of the dense optical flow

In [4]:
def dense_grid(mag, ang):

    mag = mag*3
        
    sampling = 20

    sam_mag, sam_ang = mag[::sampling, ::sampling], ang[::sampling, ::sampling]

    empty = np.zeros(shape=(mag.shape[0], mag.shape[1], 3), dtype=np.uint8)

    # build the start grid from the actual frame size
    start = np.mgrid[0:mag.shape[0]:sampling, 0:mag.shape[1]:sampling]

    end_point = (
        np.clip(np.squeeze(start[0, ...]) + sam_mag * np.cos(sam_ang), 0, mag.shape[0]),
        np.clip(np.squeeze(start[1, ...]) - sam_mag * np.sin(sam_ang), 0, mag.shape[1])
    )

    for i in range(sam_mag.shape[0]):
        for j in range(sam_mag.shape[1]):
            cv2.line(empty, (j * sampling, i * sampling), (int(end_point[1][i, j]), int(end_point[0][i, j])), (0, 255, 0), 2)
            cv2.circle(empty, (int(end_point[1][i, j]), int(end_point[0][i, j])), 3, (0, 0, 255), -1)

    return cv2.cvtColor(empty, cv2.COLOR_BGR2RGB)


def dense_color_repr(mag, ang):
    hsv = np.zeros((*gray1.shape, 3), dtype=np.uint8)
    hsv[..., 0] = ang * 180 / np.pi / 2      # hue = direction
    hsv[..., 1] = 255                         # saturation = full
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)  # brightness = speed

    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

### Loop showing the dense optical flow

In [5]:
cap = cv2.VideoCapture('Robots.mp4') 

ret = True
previous_frame = np.array(0)
feats_history = []
history_len = 20    # how many past frames to keep drawing
frame_to_show = np.array(0)

while ret:
        ret, current_frame = cap.read() # Read every frame
        if not ret: 
                break # break if not read
        
        # Hold current frame
        if previous_frame.any():
                # Convert to grayscale
                gray1 = cv2.cvtColor(previous_frame, cv2.COLOR_RGB2GRAY)
                gray2 = cv2.cvtColor(current_frame, cv2.COLOR_RGB2GRAY)

                # Track the features
                feat1 = cv2.goodFeaturesToTrack(gray1, maxCorners=100, qualityLevel=0.3, minDistance=7)

                # watch out if no features are found
                if feat1 is None:
                    previous_frame = current_frame
                    continue

                feat2, status, error = cv2.calcOpticalFlowPyrLK(gray1, gray2, feat1, None)


                # Get the flow between the pictures
                flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 25, 3, 5, 1.5, 0)
                mag, ang = cv2.cartToPolar(flow[:,:,0], flow[:,:,1])

                # Choose how to show the dense optical flow
                frame_to_show = dense_color_repr(mag, ang)
        else:
                frame_to_show = current_frame   # nothing to compare yet, just show the raw frame

        cv2.imshow('image', frame_to_show)
        cv2.waitKey(20)

        previous_frame = current_frame

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)   # let the event loop actually process the close

KeyboardInterrupt: 